In [1]:
import os
import pandas as pd
import cv2
import numpy as np
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from matplotlib import pyplot as plt
from collections import Counter
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications import ResNet50, EfficientNetB0, InceptionV3, DenseNet121, Xception
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from script import process_df_for_resnet, plot_image_with_predictions, create_model, get_top_predictions, custom_train_loop

In [2]:
big_csv_path = "2_preprocessed_sequences_with_interpolation_v0.csv"
df_big = pd.read_csv(big_csv_path)
df_big.columns

Index(['Unnamed: 0', 'Extracted_Datetime', 'Dataset_prefix_group_id',
       'Rel_Image_Path', 'Rel_Label_Path', 'Image_basename', 'Label_basename',
       'Origin_dataset_name', 'Datetime_Str', 'Extension', 'Date', 'Time',
       'Year', 'Month', 'Day', 'Hour', 'Minute', 'Second', 'img_height',
       'img_width', 'has_label', 'yolo_bbox_xcenter', 'yolo_bbox_ycenter',
       'yolo_bbox_width', 'yolo_bbox_height', 'new_label', 'pred',
       'nb_detections'],
      dtype='object')

In [3]:
csv_path = "shuffled_DS_fp_subset_enlarged_classes.csv"
df = pd.read_csv(csv_path)

In [4]:
df["Image_basename"] = [os.path.basename(path) for path in df.img_rel_path]
df.head()

,img_rel_path,label_rel_path,has_label_file,category_label,detection_label,environment_label,Image_basename
0,images/pyronear_salaunes_1_1_2023_08_09T18_11_...,labels/pyronear_salaunes_1_1_2023_08_09T18_11_...,True,nuagesbas,['nuagesbas'],[],pyronear_salaunes_1_1_2023_08_09T18_11_59.jpg
1,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg
2,images/pyronear_brison_3_2023_08_29T06_13_44.jpg,labels/pyronear_brison_3_2023_08_29T06_13_44.txt,True,montagne,['montagne'],[],pyronear_brison_3_2023_08_29T06_13_44.jpg
3,images/pyronear_valbonne_2_2023_11_05T15_54_38...,labels/pyronear_valbonne_2_2023_11_05T15_54_38...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_2_2023_11_05T15_54_38.jpg
4,images/pyronear_marguerite_1_2023_09_16T12_39_...,labels/pyronear_marguerite_1_2023_09_16T12_39_...,True,nuagesbas,['nuagesbas'],[],pyronear_marguerite_1_2023_09_16T12_39_29.jpg


In [5]:
df_bbox = df_big.copy()
df_bbox = df_bbox[['Image_basename', 'yolo_bbox_xcenter', 'yolo_bbox_ycenter','yolo_bbox_width', 'yolo_bbox_height', 'nb_detections']]
df_bbox.head()

,Image_basename,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,nb_detections
0,ADF_1320_2023_05_23T17_18_31.jpg,0.443542,0.329634,0.024021,0.036620,1.0
1,ADF_1320_2023_05_23T17_19_01.jpg,0.445935,0.332069,0.016495,0.024454,1.0
2,ADF_1320_2023_05_23T17_19_31.jpg,0.435669,0.335111,0.038391,0.032963,1.0
3,ADF_1320_2023_05_23T17_20_01.jpg,0.443198,0.327810,0.017865,0.028102,1.0
4,ADF_1320_2023_05_23T17_20_01.jpg,0.443198,0.327810,0.017865,0.028102,1.0


In [6]:
final_df = pd.merge(df, df_bbox, on='Image_basename', how='inner')
final_df

,img_rel_path,label_rel_path,has_label_file,category_label,detection_label,environment_label,Image_basename,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,nb_detections
0,images/pyronear_salaunes_1_1_2023_08_09T18_11_...,labels/pyronear_salaunes_1_1_2023_08_09T18_11_...,True,nuagesbas,['nuagesbas'],[],pyronear_salaunes_1_1_2023_08_09T18_11_59.jpg,0.977803,0.470367,0.044394,0.115933,1.0
1,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
2,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
3,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
4,images/pyronear_brison_3_2023_08_29T06_13_44.jpg,labels/pyronear_brison_3_2023_08_29T06_13_44.txt,True,montagne,['montagne'],[],pyronear_brison_3_2023_08_29T06_13_44.jpg,0.451882,0.597144,0.054057,0.048884,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12318,images/pyronear_salaunes_2_1_2023_08_28T18_29_...,labels/pyronear_salaunes_2_1_2023_08_28T18_29_...,True,grosnuages,['grosnuagesrefletssoleil'],[],pyronear_salaunes_2_1_2023_08_28T18_29_29.jpg,0.155722,0.203692,0.311443,0.404528,1.0
12319,images/pyronear_valbonne_4_2023_10_30T15_35_00...,labels/pyronear_valbonne_4_2023_10_30T15_35_00...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_4_2023_10_30T15_35_00.jpg,0.408868,0.555045,0.018049,0.024031,1.0
12320,images/pyronear_valbonne_4_2023_11_02T08_56_06...,labels/pyronear_valbonne_4_2023_11_02T08_56_06...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_4_2023_11_02T08_56_06.jpg,0.424089,0.586755,0.043256,0.043289,1.0
12321,images/pyronear_ferion_2_2023_10_17T14_34_06.jpg,labels/pyronear_ferion_2_2023_10_17T14_34_06.txt,True,nuagesbas,['nuagesbas'],['brume'],pyronear_ferion_2_2023_10_17T14_34_06.jpg,0.964625,0.520038,0.070750,0.197814,1.0


In [9]:
X, y, category_mapping = process_df_for_resnet(final_df.head(), min_num=1, label_cat="arbre")
y

[0, 0, 0, 0, 0]

In [10]:
final_df.head()

,img_rel_path,label_rel_path,has_label_file,category_label,detection_label,environment_label,Image_basename,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,nb_detections
0,images/pyronear_salaunes_1_1_2023_08_09T18_11_...,labels/pyronear_salaunes_1_1_2023_08_09T18_11_...,True,nuagesbas,['nuagesbas'],[],pyronear_salaunes_1_1_2023_08_09T18_11_59.jpg,0.977803,0.470367,0.044394,0.115933,1.0
1,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
2,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
3,images/pyronear_courmettes_1_2023_11_07T14_03_...,labels/pyronear_courmettes_1_2023_11_07T14_03_...,True,nuagesbas,['nuagesbas'],[],pyronear_courmettes_1_2023_11_07T14_03_17.jpg,0.160458,0.491901,0.139662,0.065403,1.0
4,images/pyronear_brison_3_2023_08_29T06_13_44.jpg,labels/pyronear_brison_3_2023_08_29T06_13_44.txt,True,montagne,['montagne'],[],pyronear_brison_3_2023_08_29T06_13_44.jpg,0.451882,0.597144,0.054057,0.048884,1.0


In [11]:
label_category= "arbre"

In [12]:
final_df[7500:]

,img_rel_path,label_rel_path,has_label_file,category_label,detection_label,environment_label,Image_basename,yolo_bbox_xcenter,yolo_bbox_ycenter,yolo_bbox_width,yolo_bbox_height,nb_detections
7500,images/pyronear_valbonne_2_2023_11_05T14_36_05...,labels/pyronear_valbonne_2_2023_11_05T14_36_05...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_2_2023_11_05T14_36_05.jpg,0.911933,0.452307,0.170974,0.054950,2.0
7501,images/pyronear_valbonne_2_2023_11_05T14_36_05...,labels/pyronear_valbonne_2_2023_11_05T14_36_05...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_2_2023_11_05T14_36_05.jpg,0.293748,0.432780,0.044441,0.024060,2.0
7502,images/pyronear_marguerite_4_2023_10_18T09_09_...,labels/pyronear_marguerite_4_2023_10_18T09_09_...,True,batiment,['batiment'],"['brume', 'pluie']",pyronear_marguerite_4_2023_10_18T09_09_50.jpg,0.074944,0.720923,0.149888,0.554707,1.0
7503,images/pyronear_cabanelle_2_2023_09_30T05_49_4...,labels/pyronear_cabanelle_2_2023_09_30T05_49_4...,True,soleil,"['glare', 'refletslac', 'soleil', 'tacherouge']",[],pyronear_cabanelle_2_2023_09_30T05_49_40.jpg,0.383801,0.192537,0.219428,0.383472,1.0
7504,images/pyronear_valbonne_4_2023_10_02T05_50_27...,labels/pyronear_valbonne_4_2023_10_02T05_50_27...,True,soleil,['soleil'],['batiment'],pyronear_valbonne_4_2023_10_02T05_50_27.jpg,0.330139,0.565983,0.009784,0.046927,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...
12318,images/pyronear_salaunes_2_1_2023_08_28T18_29_...,labels/pyronear_salaunes_2_1_2023_08_28T18_29_...,True,grosnuages,['grosnuagesrefletssoleil'],[],pyronear_salaunes_2_1_2023_08_28T18_29_29.jpg,0.155722,0.203692,0.311443,0.404528,1.0
12319,images/pyronear_valbonne_4_2023_10_30T15_35_00...,labels/pyronear_valbonne_4_2023_10_30T15_35_00...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_4_2023_10_30T15_35_00.jpg,0.408868,0.555045,0.018049,0.024031,1.0
12320,images/pyronear_valbonne_4_2023_11_02T08_56_06...,labels/pyronear_valbonne_4_2023_11_02T08_56_06...,True,nuagesbas,['nuagesbas'],[],pyronear_valbonne_4_2023_11_02T08_56_06.jpg,0.424089,0.586755,0.043256,0.043289,1.0
12321,images/pyronear_ferion_2_2023_10_17T14_34_06.jpg,labels/pyronear_ferion_2_2023_10_17T14_34_06.txt,True,nuagesbas,['nuagesbas'],['brume'],pyronear_ferion_2_2023_10_17T14_34_06.jpg,0.964625,0.520038,0.070750,0.197814,1.0


In [13]:
X, y, category_mapping = process_df_for_resnet(final_df[7500:], min_num=1, label_cat=label_category)

too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much hors cat
too much h

In [15]:
model_name = "inceptionv3"
print("Creating model ...")
model = create_model(backbone_name=model_name, num_classes=1)

model.load_weights("test_inceptionv3_arbre_equilibrate.weights.h5")

Creating model ...


In [16]:
X[0].shape

(224, 224, 3)

In [17]:
y_predict = model.predict(np.array(X))

41/41 ━━━━━━━━━━━━━━━━━━━━ 8s 180ms/step


In [18]:
y_predict.mean()

np.float32(0.13399112)

In [19]:
y_predict.max()

np.float32(0.99687755)

In [20]:
y = np.array(y)

In [21]:
y.max()

np.int64(1)

In [22]:
y_predict_binary = [1 if x>0.5 else 0 for x in y_predict]
y_predict_binary

[0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,


In [23]:
from sklearn.metrics import classification_report, accuracy_score

result = classification_report(y, y_predict_binary)
print(result)
print(accuracy_score(y, y_predict_binary))

              precision    recall  f1-score   support

           0       0.93      0.99      0.96      1084
           1       0.95      0.60      0.73       210

    accuracy                           0.93      1294
   macro avg       0.94      0.79      0.85      1294
weighted avg       0.93      0.93      0.92      1294

0.9296754250386399
